# Stage 5 · Q1 + Q2

The first two questions of the brief's Section 6:

- **Q1** — basket size, in-basket weighting scheme, average daily turnover
- **Q2** — headline performance table at 50M / 250M / 1B AUM (under the Section 6.3 cost schedule)

**Code**: `q1.py` (hyper-parameter selection), `q2.py` (headline-table assembly).
The shared engine `stage5_portfolio.py` (data loading / metrics / QuantStats) is read-only.
Q3/Q4/Q5 are handled by the teammates' `q3.py` / `q4.py` / `q5.py`.

## Input data (from Steps 2 / 3 / 4)

| Source | File | Used for |
| --- | --- | --- |
| Step 4 XGBoost2 | `xgboost2_predictions_all.parquet` | realised overnight return `r_on_next` + XGB rank score + sample split |
| Step 4 Ridge | `ridge_oos_predictions.parquet` | Ridge out-of-sample score (valid 2019–21 + test 2022–24) |
| Step 2 | `stage2_capacity_eligibility.parquet` | trade-eligible flag + per-AUM participation cap (5%·ADV20) |
| Step 3 | `step3_borrow_tiers.parquet` | daily short borrow cost + tier |

> Honest out-of-sample window = 2019–2024 (both models have OOS scores there). 2010–2018 is the XGBoost in-sample training period and is not used for the headline.

In [1]:
import warnings; warnings.filterwarnings('ignore')
import numpy as np, pandas as pd
import stage5_portfolio as s5                                                    # shared engine (data/metrics/reporting)
from q1 import assert_config_in_sync, basket_turnover_table, select_hyperparams  # Q1 logic
from q2 import headline_table, ensemble_robustness                              # Q2 logic
pd.set_option('display.width', 200); pd.set_option('display.max_columns', None)

cfg = s5.Step5Config()       # fixed Section 6.3 cost schedule + the three AUM levels
panel = s5.load_panel(cfg)   # merge the four sources into one point-in-time panel
print('panel rows:', len(panel), '| dates:', panel.date.min().date(), '->', panel.date.max().date())

panel rows: 3170073 | dates: 2010-03-03 -> 2024-12-30


## Q1 · basket size / weighting scheme / average turnover

**Basket size**: take a top/bottom `quantile` of the cross-section for the long/short legs.
Together with the ensemble weight `w_ridge`, it is searched **jointly by net Sharpe on the
validation split (2019–2021) only** (the test set is never used for tuning).

In [2]:
# Joint (w_ridge, quantile) search on validation -- logic lives in q1.py
w_best, q_best, grid = select_hyperparams(panel, cfg)
assert_config_in_sync(w_best, q_best)
cfg.w_ridge, cfg.quantile = w_best, q_best
print(f'selected on validation: w_ridge={w_best}, quantile={q_best}')
grid.pivot(index='w_ridge', columns='quantile', values='valid_net_sharpe').round(2)

selected on validation: w_ridge=0.3, quantile=0.02


quantile,0.02,0.03,0.05,0.10
w_ridge,,,,
0.0,0.91,0.65,0.46,-0.16
0.3,1.00,0.84,0.27,-0.07
0.5,0.96,0.70,0.31,-0.21
0.7,0.53,0.47,0.08,-0.30
1.0,-0.50,-0.74,-0.93,-1.01


> The validation net-Sharpe maximum is at `w_ridge=0.3, quantile=0.02`.
> Intuition: a decile (0.10) dilutes the long/short spread to ~2.7 bps/day < the 4 bps/day
> round-trip cost → net return must be negative; only concentrating the book on the
> highest-conviction extreme scores (~2%) lets gross return clear the cost. This is the
> core trade-off of the strategy.

**Weighting scheme**: equal weight inside each basket, then iterative pro-rata
redistribution under the `5%·ADV20` per-name cap via `water_fill` (Section 6.2).
Gross = 100%·AUM (50% long / 50% short), strictly dollar-neutral.

In [3]:
# Q1 basket + turnover: call q1.py's basket_turnover_table (single source; identical to `python q1.py`)
# XGBoost-only = headline strategy; OOS 2019-2024 = honest reporting window.
q1 = basket_turnover_table(panel, cfg)
q1.to_csv('Stage_5_portfolio_outputs/stage5_q1_basket_turnover.csv', index=False)
q1

,aum,model,window,quantile,median_names_per_side,min_names_per_side,max_names_per_side,weighting,avg_daily_turnover,avg_gross_exposure
0,50000000,xgboost_only,oos_2019_2024,0.02,19,14,20,equal weight + participation-cap redistribution,0.999885,0.999885
1,250000000,xgboost_only,oos_2019_2024,0.02,19,14,20,equal weight + participation-cap redistribution,0.900450,0.900450
2,1000000000,xgboost_only,oos_2019_2024,0.02,19,14,20,equal weight + participation-cap redistribution,0.503086,0.503086


> Turnover falls as AUM rises: 50M is near fully turned over (~1.00), 250M ~0.90, 1B ~0.50.
> This is the capacity effect: at larger AUM the `5%·ADV20` participation cap binds more often,
> so the book cannot always absorb the target gross exposure.
> The realised OOS basket holds a median of 19 names per side (daily range 14–20).

## Q2 · headline performance table (50M / 250M / 1B)

**Model choice**: the headline uses the **XGBoost-only score** — it is the only score with
full 2010–2024 coverage (Ridge exists only for 2019–2024).

- `full_2010_2024`: satisfies the brief's literal "2010–2024" request, but **2010–2018 is
  in-sample** (`contains_insample=True`) → inflated, must be flagged in the report.
- `oos_2019_2024` / `test_2022_2024`: the honest out-of-sample figures.

The Ridge+XGB **ensemble is reported separately as an OOS robustness check** (second table
below), not in the headline.

In [4]:
# headline (XGBoost-only, full window + OOS) -- logic lives in q2.py
table, store = headline_table(panel, cfg)
table.to_csv('Stage_5_portfolio_outputs/stage5_headline_performance.csv')
display(table[['model', 'contains_insample', 'basket_quantile', 'cost_round_trip_bps',
               'n_days', 'net_ann_return', 'net_ann_vol', 'net_sharpe',
               'gross_sharpe', 'max_drawdown', 'avg_daily_turnover',
               'avg_gross_exposure', 'frac_days_at_cap', 'borrow_sharpe_drag']].round(3))

# ensemble as OOS robustness only (Ridge+XGB vs XGB-only), not the headline table.
robust = ensemble_robustness(panel, cfg)
robust.to_csv('Stage_5_portfolio_outputs/stage5_ensemble_robustness.csv')
print('\nEnsemble robustness (OOS only) -- net Sharpe:')
display(robust['net_sharpe'].unstack('model').round(2))



model  contains_insample  basket_quantile  cost_round_trip_bps  n_days  net_ann_return  net_ann_vol  net_sharpe  gross_sharpe  max_drawdown  avg_daily_turnover  \
window         aum                                                                                                                                                                                  
full_2010_2024 50000000    xgboost_only               True             0.02                  4.0    3733           0.180        0.063       2.850         4.512        -0.225               0.992   
               250000000   xgboost_only               True             0.02                  4.0    3733           0.156        0.064       2.454         3.712        -0.167               0.772   
               1000000000  xgboost_only               True             0.02                  4.0    3733           0.067        0.051       1.315         2.044        -0.194               0.360   
oos_2019_2024  50000000    xgboost_only              False             0.02                  4.0    1509           0.013        0.083       0.157         1.413        -0.225               1.000   
               250000000   xgboost_only              False             0.02                  4.0    1509           0.041        0.087       0.466         1.538        -0.167               0.900   
               1000000000  xgboost_only              False             0.02                  4.0    1509           0.019        0.076       0.246         0.934        -0.194               0.503   
test_2022_2024 50000000    xgboost_only              False             0.02                  4.0     752          -0.021        0.076      -0.277         1.098        -0.145               1.000   
               250000000   xgboost_only              False             0.02                  4.0     752          -0.003        0.082      -0.036         1.146        -0.107               0.937   
               1000000000  xgboost_only              False             0.02                  4.0     752          -0.036        0.065      -0.556         0.316        -0.119               0.542   

                           avg_gross_exposure  frac_days_at_cap  borrow_sharpe_drag  
window         aum                                                                   
full_2010_2024 50000000                 0.992             0.037               0.078  
               250000000                0.772             0.494               0.043  
               1000000000               0.360             0.877               0.026  
oos_2019_2024  50000000                 1.000             0.002               0.045  
               250000000                0.900             0.276               0.034  
               1000000000               0.503             0.794               0.022  
test_2022_2024 50000000                 1.000             0.000               0.042  
               250000000                0.937             0.210               0.033  
               1000000000               0.542             0.767               0.025


Ensemble robustness (OOS only) -- net Sharpe:


model                      ridge_xgb_ensemble  xgboost_only
window         aum                                         
oos_2019_2024  50000000                  0.07          0.16
               250000000                 0.42          0.47
               1000000000                0.43          0.25
test_2022_2024 50000000                 -0.30         -0.28
               250000000                -0.23         -0.04
               1000000000               -0.34         -0.56